# Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

## Lab 3 — Benchmark Evaluation

In this notebook, we evaluate the SFT fine-tuned model using a **benchmark evaluation** and compare it against the base model.

### What you'll do

1. Retrieve the fine-tuned model from the Model Registry
2. Explore available benchmarks and create a `BenchMarkEvaluator`
3. Run evaluation on both the fine-tuned **and** base model
4. Compare the results

---

## 1. Prerequisites

### Set up the SageMaker session

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Retrieve the fine-tuned model

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

base_model_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_version= "*"
model_package_group_name = f"{base_model_id}-mpg"
model_package_version = "1"

model_package_group = ModelPackageGroup.get(model_package_group_name)

fine_tuned_model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/benchmark-evaluation"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}/benchmark-evaluation"


if default_prefix:
    output_path_base = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/benchmark-evaluation-base"
else:
    output_path_base = f"s3://{bucket_name}/{base_model_id}/benchmark-evaluation-base"

print(f"Evaluation base output path: {output_path_base}")
print(f"Evaluation output path: {output_path}")

In [ ]:
# Get BaseModel info from the fine-tuned model package
ft_response = sm_client.describe_model_package(
    ModelPackageName=fine_tuned_model_package_arn
)
base_model_info = ft_response['InferenceSpecification']['Containers'][0]['BaseModel']

print(base_model_info)
# {'HubContentName': 'huggingface-llm-qwen2-5-7b-instruct', 'HubContentVersion': '1.27.0', 'RecipeName': 'llmft_qwen2_5_7b_seq4k_gpu_sft_lora'}

In [ ]:
import boto3


base_model_id = "huggingface-llm-qwen2-5-7b-instruct"
base_mpg_name = f"{base_model_id}-base-mpg"

# Step 1: Create model package group
sm_client.create_model_package_group(
    ModelPackageGroupName=base_mpg_name,
    ModelPackageGroupDescription="Base model package group for Qwen2.5-7B-Instruct"
)
print(f"Created model package group: {base_mpg_name}")

# Step 2: Register the base model as a model package
response = sm_client.create_model_package(
    ModelPackageGroupName=base_mpg_name,
    ModelPackageDescription="Qwen2.5-7B-Instruct base model",
    InferenceSpecification={
        'Containers': [{
            'ModelDataSource': {
                'S3DataSource': {
                    'S3Uri': 's3://jumpstart-cache-prod-us-east-1/huggingface-llm/huggingface-llm-qwen2-5-7b-instruct/artifacts/inference/v1.0.0/',
                    'S3DataType': 'S3Prefix',
                    'CompressionType': 'None'
                }
            },
            'BaseModel': base_model_info
        }],
        'SupportedTransformInstanceTypes': ['ml.g5.12xlarge'],
        'SupportedRealtimeInferenceInstanceTypes': ['ml.g5.12xlarge'],
        'SupportedContentTypes': ['application/json'],
        'SupportedResponseMIMETypes': ['application/json'],
    },
    ModelApprovalStatus='Approved'
)
print(f"Base Model Package ARN: {response['ModelPackageArn']}")

In [ ]:
# # Step 1: Delete all model packages in the group
# pkgs = sm_client.list_model_packages(ModelPackageGroupName=base_mpg_name)
# for pkg in pkgs['ModelPackageSummaryList']:
#     sm_client.delete_model_package(ModelPackageName=pkg['ModelPackageArn'])
#     print(f"Deleted: {pkg['ModelPackageArn']}")

# # Step 2: Delete the group
# sm_client.delete_model_package_group(ModelPackageGroupName=base_mpg_name)
# print(f"Deleted group: {base_mpg_name}")

In [ ]:
# Get model package group ARN
mpg_response = sm_client.describe_model_package_group(
    ModelPackageGroupName=base_mpg_name
)
base_model_package_group_arn = mpg_response['ModelPackageGroupArn']

# Get latest model package ARN
pkg_response = sm_client.list_model_packages(
    ModelPackageGroupName=base_mpg_name,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1
)
base_model_package_arn = pkg_response['ModelPackageSummaryList'][0]['ModelPackageArn']

print(f"Base Model Group ARN: {base_model_package_group_arn}")
print(f"Base Model Package ARN: {base_model_package_arn}")


---

## 2. Explore available benchmarks

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

Benchmark = get_benchmarks()
pprint(list(Benchmark))

Inspect the benchmark we will use. Change `Benchmark.MEDICAL` to whichever benchmark is most relevant from the list above (e.g. `Benchmark.MMLU`, `Benchmark.MATH`, etc.).

In [ ]:
# Pick the benchmark that best fits your use case from the list above.
# For a medical SFT model, look for a medical benchmark (e.g. MEDICAL, MMLU).
# If no medical-specific benchmark exists, MMLU is a good general choice.
SELECTED_BENCHMARK = Benchmark.BBH

pprint(get_benchmark_properties(benchmark=SELECTED_BENCHMARK))

---

## 3. Run benchmark evaluation

We set `evaluate_base_model=True` so SageMaker evaluates both the fine-tuned model and the original Qwen 2.5 7B Instruct base model side by side.

In [ ]:
evaluator = BenchMarkEvaluator(
    benchmark=SELECTED_BENCHMARK,
    model=fine_tuned_model_package_arn,
    model_package_group=model_package_group_name,
    base_eval_name='fine-tuned-model-sft',
    s3_output_path=output_path,
    evaluate_base_model=False,
)

execution = evaluator.evaluate()
#execution.wait()


base_evaluator = BenchMarkEvaluator(
    benchmark=SELECTED_BENCHMARK,
    model=base_model_package_arn,
    model_package_group=base_model_package_group_arn,
    base_eval_name='base-model-sft',
    s3_output_path=output_path_base,
    evaluate_base_model=False,
)

execution_base = base_evaluator.evaluate()
execution_base.wait()

In [ ]:
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

# Get all succeeded evaluations and take the first 2
all_succeeded = [
    e for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
    if e.status.overall_status == "Succeeded"
]

last_two_succeeded = all_succeeded[:2]

# Print both
for i, execution in enumerate(last_two_succeeded, 1):
    print(f"=== Succeeded Evaluation #{i} ===")
    pprint(execution)
    pprint(execution.show_results())

---

## 4. View evaluation results

In [ ]:
from sagemaker.train.common_utils import show_results_utils
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

latest_succeeded = next(
    (
        e
        for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
        if e.status.overall_status == "Succeeded"
    ),
    None,
)
pprint(latest_succeeded)

In [ ]:
if latest_succeeded:
    latest_succeeded.show_results()

### Download and visualize results

Download the detailed results JSON from S3 for both the custom (fine-tuned) and base model evaluations.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from urllib.parse import urlparse

parsed = urlparse(latest_succeeded.s3_output_path)
bucket = parsed.netloc
prefix = parsed.path.lstrip("/")
pipeline_name = latest_succeeded.name

os.makedirs("./tmp", exist_ok=True)


def download_results(step_suffix):
    """Download the results JSON for a given pipeline step."""
    resp = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=f"{prefix}/pipelines-{pipeline_name}-{step_suffix}",
    )
    key = next(
        obj["Key"]
        for obj in resp.get("Contents", [])
        if obj["Key"].endswith(".json") and "results_" in obj["Key"]
    )
    local = f"./tmp/{step_suffix}.json"
    s3_client.download_file(bucket, key, local)
    with open(local) as f:
        return json.load(f)


custom_results = download_results("EvaluateCustomModel")
base_results = download_results("EvaluateBaseModel")

print("Custom model results:")
print(json.dumps(custom_results["results"].get("all", {}), indent=2))
print("\nBase model results:")
print(json.dumps(base_results["results"].get("all", {}), indent=2))

In [ ]:
# MMLU medical-related subcategories to filter for
MEDICAL_KEYWORDS = {
    "clinical_knowledge", "medical_genetics", "anatomy",
    "professional_medicine", "college_medicine", "college_biology",
    "nutrition", "virology", "human_aging",
}


def parse_results(results, label, medical_only=True):
    """Parse benchmark results JSON into a DataFrame, optionally filtering to medical categories."""
    rows = []
    for key, scores in results["results"].items():
        if key == "all" or "_average" in key:
            continue
        raw_category = key.split(":")[1].split("|")[0]
        if medical_only and raw_category not in MEDICAL_KEYWORDS:
            continue
        category = raw_category.replace("_", " ").title()
        score_val = next((v for v in scores.values() if isinstance(v, (int, float))), 0)
        rows.append({"category": category, "score": score_val, "model": label})
    return pd.DataFrame(rows)


df_custom = parse_results(custom_results, "Fine-tuned")
df_base = parse_results(base_results, "Base")
df = pd.concat([df_custom, df_base], ignore_index=True)

print(f"Filtered to {len(df_custom)} medical categories")

# Side-by-side bar chart
pivot = df.pivot(index="category", columns="model", values="score").sort_values("Fine-tuned")

fig, ax = plt.subplots(figsize=(10, max(6, len(pivot) * 0.5)))
pivot.plot.barh(ax=ax, color={"Base": "#aaa", "Fine-tuned": "steelblue"})
ax.set_xlabel("Score")
ax.set_title("MMLU Medical Categories: Base vs Fine-tuned Model")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()